# Brain Tumor MRI — WGAN-GP Full Pipeline
## Dataset Preprocessing · Exploratory Analysis · Training · Synthesis · Evaluation

**Section 2 — Medical Image Synthesis & Data Augmentation**  
**Dataset:** Brain Tumor MRI Dataset — 7,200 Human Brain MRI Images  
- `Training/` — glioma (1,400) · meningioma (1,400) · pituitary (1,400) · notumor (1,400)  
- `Testing/`  — glioma (400)   · meningioma (400)   · pituitary (400)   · notumor (400)  

**Architecture:** WGAN-GP — Deconvolutional Generator (nz=200) + Convolutional Critic  
**Purpose:** Generate realistic synthetic brain MRI images per tumor class for data augmentation while preserving 100% patient privacy.


## 1. Imports & Environment Setup

In [ ]:
import os, glob, random, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from collections import Counter

import tensorflow as tf
from tensorflow.keras import layers, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as kl_entropy

warnings.filterwarnings("ignore")
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {len(tf.config.list_physical_devices('GPU'))}")
print("Environment ready.")

## 2. Configuration & Hyperparameters

In [ ]:
# ── Dataset ────────────────────────────────────────────────────────────────
DATASET_ROOT = "."
TRAIN_DIR    = os.path.join(DATASET_ROOT, "Training")
TEST_DIR     = os.path.join(DATASET_ROOT, "Testing")
MRI_CLASSES  = ["glioma", "meningioma", "notumor", "pituitary"]

# ── Image ──────────────────────────────────────────────────────────────────
IMG_SIZE     = 64
IMG_CHANNELS = 1

# ── WGAN-GP Hyperparameters (from user provided Lasagne/Theano code) ───────
NZ           = 200          # Latent vector dimension
LAMBDA_GP    = 10           # Gradient penalty weight (lambda)
N_CRITIC     = 5            # Critic updates per generator update
BATCH_SIZE   = 32
EPOCHS       = 50           # Increase for better quality
LR_GEN       = 5e-5         # Adam lr generator
LR_CRIT      = 5e-5         # Adam lr critic
BETA_1       = 0.5
BETA_2       = 0.9

# ── Output dirs ────────────────────────────────────────────────────────────
WEIGHTS_DIR  = "saved_weights"
MONTAGE_DIR  = "montages"
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(MONTAGE_DIR,  exist_ok=True)

# ── Which class to train on ─────────────────────────────────────────────────
TARGET_CLASS = "glioma"     # Change to: meningioma | notumor | pituitary

print("Configuration:")
for k,v in dict(TARGET_CLASS=TARGET_CLASS,IMG_SIZE=IMG_SIZE,NZ=NZ,
                LAMBDA_GP=LAMBDA_GP,N_CRITIC=N_CRITIC,
                EPOCHS=EPOCHS,BATCH_SIZE=BATCH_SIZE).items():
    print(f"  {k:<14}: {v}")

## 3. Dataset Scanning & Exploratory Analysis
### 3.1 File Registry

In [ ]:
registry = {"Training": {}, "Testing": {}}
for split, base in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    for cls in MRI_CLASSES:
        folder = os.path.join(base, cls)
        files  = (glob.glob(os.path.join(folder, "*.jpg")) +
                  glob.glob(os.path.join(folder, "*.jpeg")) +
                  glob.glob(os.path.join(folder, "*.png")))
        registry[split][cls] = files

rows = []
for cls in MRI_CLASSES:
    tr = len(registry["Training"][cls])
    te = len(registry["Testing"][cls])
    rows.append({"Class": cls, "Training": tr, "Testing": te, "Total": tr+te})
df_summary = pd.DataFrame(rows)
df_summary.loc[len(df_summary)] = ["TOTAL",
    df_summary["Training"].sum(), df_summary["Testing"].sum(), df_summary["Total"].sum()]
print(df_summary.to_string(index=False))

### 3.2 Class Distribution Chart

In [ ]:
colors = ["#C62828","#F57F17","#1B5E20","#1A237E"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, split, alpha in zip(axes, ["Training","Testing"], [1.0, 0.7]):
    counts = [len(registry[split][c]) for c in MRI_CLASSES]
    bars   = ax.bar(MRI_CLASSES, counts, color=colors, edgecolor="black", lw=0.7, alpha=alpha)
    for b, v in zip(bars, counts):
        ax.text(b.get_x()+b.get_width()/2, v+8, str(v), ha="center", fontweight="bold")
    ax.set_title(f"{split} Set — Class Distribution", fontweight="bold")
    ax.set_ylim(0, max(counts)*1.25)
plt.suptitle("Brain Tumor MRI Dataset — Class Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Sample Real MRI Images per Class

In [ ]:
class_colors = {"glioma":"#C62828","meningioma":"#F57F17","notumor":"#1B5E20","pituitary":"#1A237E"}
fig, axes = plt.subplots(len(MRI_CLASSES), 6, figsize=(16, 11))
fig.suptitle("Real Brain MRI Samples — Training Dataset", fontsize=14, fontweight="bold")
for row, cls in enumerate(MRI_CLASSES):
    files = random.sample(registry["Training"][cls], 6)
    for col, fp in enumerate(files):
        img = Image.open(fp).convert("L").resize((IMG_SIZE, IMG_SIZE))
        axes[row, col].imshow(np.array(img), cmap="gray")
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(cls.upper(), fontsize=11,
                color=class_colors[cls], fontweight="bold",
                rotation=0, labelpad=60, va="center")
plt.tight_layout()
plt.savefig("montages/real_samples_grid.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.4 Pixel Intensity Distribution per Class

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()
for idx, (cls, color) in enumerate(zip(MRI_CLASSES, colors)):
    pxl = []
    for fp in registry["Training"][cls][:200]:
        try:
            img = Image.open(fp).convert("L").resize((IMG_SIZE, IMG_SIZE))
            pxl.extend(np.array(img).flatten().tolist())
        except: pass
    pxl = np.array(pxl, dtype=np.float32) / 255.0
    axes[idx].hist(pxl, bins=100, color=color, alpha=0.75, edgecolor="none")
    axes[idx].axvline(pxl.mean(), color="black", ls="--", lw=2,
                       label=f"Mean={pxl.mean():.3f}")
    axes[idx].set_title(f"{cls.upper()} Pixel Distribution", fontweight="bold", color=color)
    axes[idx].set_xlabel("Normalized pixel value")
    axes[idx].set_ylabel("Frequency")
    axes[idx].legend()
plt.suptitle("Pixel Intensity Distributions by Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/pixel_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.5 Mean MRI Image per Class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for idx, cls in enumerate(MRI_CLASSES):
    frames = []
    for fp in registry["Training"][cls][:400]:
        try:
            img = Image.open(fp).convert("L").resize((IMG_SIZE, IMG_SIZE))
            frames.append(np.array(img, dtype=np.float32))
        except: pass
    mean_img = np.mean(frames, axis=0)
    im = axes[idx].imshow(mean_img, cmap="hot", vmin=0, vmax=255)
    axes[idx].set_title(f"{cls.upper()}\n(mean of {len(frames)} imgs)",
                         fontweight="bold")
    axes[idx].axis("off")
    plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
plt.suptitle("Mean Pixel Intensity per Tumor Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/mean_images.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Dataset Preprocessing
### 4.1 Load & Normalize Images

In [ ]:
def load_class_images(cls, split="Training", max_images=1400, verbose=True):
    """
    Load MRI images for one class.
    Returns np.ndarray (N, IMG_SIZE, IMG_SIZE, 1) in [0, 1].
    """
    files = registry[split][cls][:]
    random.shuffle(files)
    files = files[:max_images]
    imgs  = []
    for fp in files:
        try:
            img = Image.open(fp).convert("L").resize(
                (IMG_SIZE, IMG_SIZE), Image.BILINEAR)
            imgs.append(np.array(img, dtype=np.float32) / 255.0)
        except Exception as e:
            if verbose: print(f"  Warning: {fp}: {e}")
    data = np.stack(imgs, axis=0)[:, :, :, np.newaxis]   # (N,64,64,1)
    if verbose:
        print(f"Loaded {len(data)} {split}/{cls} images")
        print(f"  Shape: {data.shape}  |  Min: {data.min():.3f}  Max: {data.max():.3f}")
        print(f"  Mean : {data.mean():.4f}  |  Std: {data.std():.4f}")
    return data

real_images = load_class_images(TARGET_CLASS, split="Training")

### 4.2 Build tf.data.Dataset Pipeline

In [ ]:
def build_tf_dataset(images, batch_size=BATCH_SIZE, shuffle_buffer=1000):
    ds = tf.data.Dataset.from_tensor_slices(images.astype(np.float32))
    ds = ds.shuffle(shuffle_buffer)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = build_tf_dataset(real_images)
for batch in train_ds.take(1):
    print(f"Batch shape : {batch.shape}")
    print(f"Min / Max   : {batch.numpy().min():.4f} / {batch.numpy().max():.4f}")

## 5. WGAN-GP Model Architecture
### 5.1 Generator (nz=200 — from user Lasagne/Theano code)
```
z ∈ R^200
  Dense(1024×4×4) → Reshape(4,4,1024)
  Conv2DTranspose(512, 4×4, s=2) + BN + ReLU  →  8×8×512
  Conv2DTranspose(256, 4×4, s=2) + BN + ReLU  → 16×16×256
  Conv2DTranspose(128, 4×4, s=2) + BN + ReLU  → 32×32×128
  Conv2DTranspose(  1, 4×4, s=2) + Sigmoid    → 64×64×1
```


In [ ]:
def build_generator(nz=NZ):
    inp = layers.Input(shape=(nz,))
    x   = layers.Dense(1024*4*4, use_bias=False)(inp)
    x   = layers.Reshape((4, 4, 1024))(x)
    # Block 1: 4→8
    x   = layers.Conv2DTranspose(512, 4, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 2: 8→16
    x   = layers.Conv2DTranspose(256, 4, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 3: 16→32
    x   = layers.Conv2DTranspose(128, 4, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    # Block 4: 32→64
    x   = layers.Conv2DTranspose(1, 4, strides=2, padding="same", activation="sigmoid")(x)
    return models.Model(inp, x, name="WGANGP_Generator")

generator = build_generator()
generator.summary()

### 5.2 Critic / Discriminator
```
x ∈ R^{64×64×1}
  Conv2D(128,  5×5, s=2) + BN + LReLU(0.2) → 32×32×128
  Conv2D(256,  5×5, s=2) + BN + LReLU(0.2) → 16×16×256
  Conv2D(512,  5×5, s=2) + BN + LReLU(0.2) →  8×8×512
  Conv2D(1024, 5×5, s=2) + BN + LReLU(0.2) →  4×4×1024
  Flatten → Dense(1)   [no sigmoid — Wasserstein output]
```


In [ ]:
def build_critic(img_shape=(IMG_SIZE, IMG_SIZE, IMG_CHANNELS)):
    inp = layers.Input(shape=img_shape)
    x   = layers.Conv2D(128,  5, strides=2, padding="same", use_bias=False)(inp)
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Conv2D(256,  5, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Conv2D(512,  5, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Conv2D(1024, 5, strides=2, padding="same", use_bias=False)(x)
    x   = layers.BatchNormalization()(x); x = layers.LeakyReLU(0.2)(x)
    x   = layers.Flatten()(x)
    x   = layers.Dense(1)(x)   # Wasserstein — no sigmoid
    return models.Model(inp, x, name="WGANGP_Critic")

critic = build_critic()
critic.summary()

### 5.3 Optimizers

In [ ]:
# Adam (lr=5e-5, beta_1=0.5, beta_2=0.9) — exact user code hyperparameters
g_optimizer = tf.keras.optimizers.Adam(learning_rate=LR_GEN,  beta_1=BETA_1, beta_2=BETA_2)
d_optimizer = tf.keras.optimizers.Adam(learning_rate=LR_CRIT, beta_1=BETA_1, beta_2=BETA_2)
print(f"G optimizer: Adam(lr={LR_GEN}, b1={BETA_1}, b2={BETA_2})")
print(f"D optimizer: Adam(lr={LR_CRIT}, b1={BETA_1}, b2={BETA_2})")

## 6. WGAN-GP Training
### 6.1 Gradient Penalty
$$\text{GP} = \lambda \cdot \mathbb{E}\left[\left(\|\nabla_{\hat{x}} D(\hat{x})\|_2 - 1\right)^2\right], \quad \hat{x} = \alpha x_\text{real} + (1-\alpha) x_\text{fake}$$


In [ ]:
@tf.function
def compute_gradient_penalty(critic, real_imgs, fake_imgs, lam=LAMBDA_GP):
    batch = tf.shape(real_imgs)[0]
    alpha = tf.random.uniform([batch, 1, 1, 1], 0.0, 1.0)
    interp = alpha * real_imgs + (1.0 - alpha) * fake_imgs
    with tf.GradientTape() as tape:
        tape.watch(interp)
        pred = critic(interp, training=True)
    grads      = tape.gradient(pred, interp)
    grad_norm  = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=[1,2,3]) + 1e-12)
    return lam * tf.reduce_mean(tf.square(grad_norm - 1.0))
print("Gradient penalty defined.")

### 6.2 Train Step Functions

In [ ]:
@tf.function
def train_critic_step(real_batch):
    """Critic loss = E[D(fake)] - E[D(real)] + GP"""
    n = tf.shape(real_batch)[0]
    z = tf.random.normal([n, NZ])
    with tf.GradientTape() as tape:
        fake        = generator(z, training=True)
        real_score  = critic(real_batch, training=True)
        fake_score  = critic(fake,       training=True)
        gp          = compute_gradient_penalty(critic, real_batch, fake)
        d_loss      = tf.reduce_mean(fake_score) - tf.reduce_mean(real_score) + gp
    grads = tape.gradient(d_loss, critic.trainable_variables)
    d_optimizer.apply_gradients(zip(grads, critic.trainable_variables))
    return d_loss, gp

@tf.function
def train_generator_step(batch_size):
    """Generator loss = -E[D(G(z))]"""
    z = tf.random.normal([batch_size, NZ])
    with tf.GradientTape() as tape:
        fake       = generator(z, training=True)
        fake_score = critic(fake, training=True)
        g_loss     = -tf.reduce_mean(fake_score)
    grads = tape.gradient(g_loss, generator.trainable_variables)
    g_optimizer.apply_gradients(zip(grads, generator.trainable_variables))
    return g_loss

print("Train step functions compiled.")

### 6.3 Montage Helper (matches `create_montage` from user code)

In [ ]:
def create_montage(gen, nz=NZ, grid=10, save_path=None):
    """10x10 grid of 100 synthetic MRI images."""
    n      = grid * grid
    z      = tf.random.normal([n, nz])
    imgs   = gen(z, training=False).numpy()[:, :, :, 0]
    h, w   = imgs.shape[1], imgs.shape[2]
    canvas = np.zeros((grid*h, grid*w), dtype=np.float32)
    for idx in range(n):
        r, c = divmod(idx, grid)
        canvas[r*h:(r+1)*h, c*w:(c+1)*w] = imgs[idx]
    if save_path:
        plt.imsave(save_path, canvas, cmap="gray", vmin=0, vmax=1)
    return canvas

def show_montage(canvas, title="Synthetic Brain MRI — 10x10 Montage", epoch=None):
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(canvas, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{title}" + (f" (Epoch {epoch})" if epoch else ""), fontsize=13, fontweight="bold")
    ax.axis("off")
    plt.tight_layout(); plt.show()

canvas0 = create_montage(generator, save_path="montages/montage_epoch_000.png")
show_montage(canvas0, epoch=0)
print("Untrained (random) montage shown.")

### 6.4 Full Training Loop

In [ ]:
g_loss_hist, d_loss_hist, gp_hist = [], [], []
spe = len(real_images) // BATCH_SIZE
print(f"Training: class={TARGET_CLASS}  epochs={EPOCHS}  steps/epoch={spe}  n_critic={N_CRITIC}")
print("-"*60)

for epoch in range(1, EPOCHS+1):
    np.random.shuffle(real_images)
    ep_d, ep_g, ep_gp = [], [], []
    t0 = time.time()

    for step in range(spe):
        i0    = step * BATCH_SIZE
        batch = tf.constant(real_images[i0:i0+BATCH_SIZE])
        # Critic: N_CRITIC steps
        for _ in range(N_CRITIC):
            d_loss, gp_val = train_critic_step(batch)
        ep_d.append(float(d_loss)); ep_gp.append(float(gp_val))
        # Generator: 1 step
        g_loss = train_generator_step(BATCH_SIZE)
        ep_g.append(float(g_loss))

    avg_d, avg_g, avg_gp = np.mean(ep_d), np.mean(ep_g), np.mean(ep_gp)
    g_loss_hist.append(avg_g)
    d_loss_hist.append(avg_d)
    gp_hist.append(avg_gp)

    if epoch % 10 == 0 or epoch == 1 or epoch == EPOCHS:
        path = f"montages/montage_epoch_{epoch:03d}.png"
        show_montage(create_montage(generator, save_path=path), epoch=epoch)

    print(f"Epoch {epoch:3d}/{EPOCHS} | D={avg_d:7.4f} | G={avg_g:7.4f} | GP={avg_gp:6.4f} | {time.time()-t0:.1f}s")

print("-"*60)
print("Training complete!")

### 6.5 Save Weights

In [ ]:
gp = os.path.join(WEIGHTS_DIR, f"gen_{TARGET_CLASS}.weights.h5")
cp = os.path.join(WEIGHTS_DIR, f"critic_{TARGET_CLASS}.weights.h5")
generator.save_weights(gp)
critic.save_weights(cp)
print(f"Generator weights : {gp}")
print(f"Critic weights    : {cp}")
print("Weights auto-loaded by app.py on the next request.")

## 7. Loss Curves

In [ ]:
ep_range = range(1, len(g_loss_hist)+1)
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
axes[0].plot(ep_range, g_loss_hist, color="#8F4E00", lw=2)
axes[0].set_title("Generator Loss G(z)",       fontweight="bold"); axes[0].grid(alpha=0.3)
axes[1].plot(ep_range, d_loss_hist, color="#C62828", lw=2)
axes[1].set_title("Critic Loss (Wasserstein)",  fontweight="bold"); axes[1].grid(alpha=0.3)
axes[2].plot(ep_range, gp_hist,    color="#1B5E20", lw=2)
axes[2].set_title(f"Gradient Penalty (λ={LAMBDA_GP})", fontweight="bold"); axes[2].grid(alpha=0.3)
for ax in axes: ax.set_xlabel("Epoch"); ax.axhline(0, color="black", ls="--", lw=0.8, alpha=0.4)
plt.suptitle(f"WGAN-GP Training Curves — {TARGET_CLASS.upper()}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Real vs Synthetic Comparison

In [ ]:
n_compare    = 8
real_samp    = real_images[np.random.choice(len(real_images), n_compare, replace=False)]
synth_samp   = generator(tf.random.normal([n_compare, NZ]), training=False).numpy()

fig, axes = plt.subplots(2, n_compare, figsize=(2*n_compare, 5))
for col in range(n_compare):
    axes[0,col].imshow(real_samp[col,:,:,0],  cmap="gray", vmin=0, vmax=1); axes[0,col].axis("off")
    axes[1,col].imshow(synth_samp[col,:,:,0], cmap="gray", vmin=0, vmax=1); axes[1,col].axis("off")
    if col==0:
        axes[0,col].set_ylabel("REAL",  fontsize=11, fontweight="bold", color="#1B5E20",
                                rotation=0, labelpad=42, va="center")
        axes[1,col].set_ylabel("SYNTH", fontsize=11, fontweight="bold", color="#8F4E00",
                                rotation=0, labelpad=42, va="center")
plt.suptitle(f"Real vs WGAN-GP Synthetic — {TARGET_CLASS.upper()}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/real_vs_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()

## 9. Evaluation Metrics
### 9.1 Quantitative Metrics

In [ ]:
N_EVAL    = 500
real_eval = real_images[:N_EVAL, :, :, 0]
synth_eval= generator(tf.random.normal([N_EVAL, NZ]), training=False).numpy()[:,:,:,0]
r_flat    = real_eval.reshape(N_EVAL, -1)
s_flat    = synth_eval.reshape(N_EVAL, -1)

fidelity_mse = mean_squared_error(r_flat, s_flat)
cosine_sim   = float(np.diag(cosine_similarity(r_flat, s_flat)).mean())
r_hist, _    = np.histogram(r_flat.flatten(), bins=256, density=True)
s_hist, _    = np.histogram(s_flat.flatten(), bins=256, density=True)
kl_div       = float(kl_entropy(r_hist + 1e-10, s_hist + 1e-10))
diversity    = float(np.mean(np.var(s_flat, axis=0)))
cov          = float(np.mean((s_flat.max(0)-s_flat.min(0))/(r_flat.max(0)-r_flat.min(0)+1e-8)))*100
fid_approx   = (r_flat.mean()-s_flat.mean())**2 + (r_flat.std()-s_flat.std())**2

print("="*55)
print(f"  WGAN-GP Evaluation — {TARGET_CLASS.upper()}")
print("="*55)
for name, val in [("Fidelity MSE", f"{fidelity_mse:.6f}"),
                   ("Cosine Similarity", f"{cosine_sim:.4f}"),
                   ("KL Divergence", f"{kl_div:.4f}"),
                   ("Diversity (Var)", f"{diversity:.4f}"),
                   ("Coverage Ratio %", f"{cov:.2f}"),
                   ("FID approximation", f"{fid_approx:.6f}")]:
    print(f"  {name:<22}: {val}")

### 9.2 Pixel Distribution: Real vs Synthetic

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(r_flat.flatten(), bins=100, density=True, alpha=0.6, color="#1A237E", label="Real")
axes[0].hist(s_flat.flatten(), bins=100, density=True, alpha=0.6, color="#FF8F00", label="WGAN-GP")
axes[0].set_title("Pixel Distribution: Real vs Synthetic", fontweight="bold")
axes[0].set_xlabel("Normalized pixel value")
axes[0].legend()
axes[0].text(0.02,0.95,f"KL={kl_div:.4f}\nCos={cosine_sim:.4f}",
             transform=axes[0].transAxes,va="top",fontsize=9,
             bbox=dict(boxstyle="round",fc="white",alpha=0.8))

metrics_bar = {"Cosine Sim": cosine_sim, "Coverage %": cov/100}
bars = axes[1].bar(metrics_bar.keys(), metrics_bar.values(),
                    color=["#1B5E20","#1A237E"], edgecolor="black", lw=0.7)
for b, v in zip(bars, metrics_bar.values()):
    axes[1].text(b.get_x()+b.get_width()/2, v+0.01, f"{v:.3f}", ha="center", fontweight="bold")
axes[1].set_ylim(0,1.15)
axes[1].set_title("Quality Metrics", fontweight="bold")
plt.suptitle(f"Evaluation — {TARGET_CLASS.upper()}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/evaluation_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Synthetic Image Generation
### 10.1 Final 10×10 Montage

In [ ]:
final_canvas = create_montage(generator, save_path=f"montages/final_{TARGET_CLASS}.png")
show_montage(final_canvas, title=f"WGAN-GP 10x10 Montage — {TARGET_CLASS.upper()}", epoch=EPOCHS)
print(f"Saved: montages/final_{TARGET_CLASS}.png")

### 10.2 Export Synthetic Images to Disk

In [ ]:
N_EXPORT  = 50
export_dir = f"synthetic_{TARGET_CLASS}"
os.makedirs(export_dir, exist_ok=True)
z_export   = tf.random.normal([N_EXPORT, NZ])
gen_imgs   = generator(z_export, training=False).numpy()
for i in range(N_EXPORT):
    arr = (gen_imgs[i,:,:,0]*255).clip(0,255).astype(np.uint8)
    Image.fromarray(arr,"L").save(os.path.join(export_dir,f"synth_{TARGET_CLASS}_{i+1:04d}.png"))
print(f"Exported {N_EXPORT} synthetic images → {export_dir}/")

fig, axes = plt.subplots(1, 8, figsize=(16, 3))
for i, ax in enumerate(axes):
    ax.imshow(gen_imgs[i,:,:,0], cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(f"#{i+1}",fontsize=9)
plt.suptitle(f"Sample Exported Synthetic {TARGET_CLASS.upper()} MRIs", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

### 10.3 Latent Space Interpolation

In [ ]:
n_steps = 10
z_a = tf.random.normal([1, NZ]).numpy()
z_b = tf.random.normal([1, NZ]).numpy()
alphas  = np.linspace(0, 1, n_steps)
z_interp = np.array([(1-a)*z_a + a*z_b for a in alphas]).squeeze(1)
interp_imgs = generator(tf.constant(z_interp, dtype=tf.float32), training=False).numpy()[:,:,:,0]

fig, axes = plt.subplots(1, n_steps, figsize=(2*n_steps, 3))
for i, ax in enumerate(axes):
    ax.imshow(interp_imgs[i], cmap="gray", vmin=0, vmax=1); ax.axis("off")
    ax.set_title(f"a={alphas[i]:.1f}", fontsize=8)
plt.suptitle(f"Latent Space Interpolation — {TARGET_CLASS.upper()}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig("montages/latent_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Load Pre-trained Weights & Inference (No Re-training)

In [ ]:
infer_class = TARGET_CLASS   # Change to any saved class
gen_path    = os.path.join(WEIGHTS_DIR, f"gen_{infer_class}.weights.h5")

if os.path.exists(gen_path):
    gen_inf = build_generator()
    _       = gen_inf(tf.zeros([1, NZ]), training=False)   # Build weights
    gen_inf.load_weights(gen_path)
    print(f"Loaded: {gen_path}")
    canvas = create_montage(gen_inf, save_path=f"montages/inference_{infer_class}.png")
    show_montage(canvas, title=f"[Inference Only] {infer_class.upper()} MRI Montage")
else:
    print(f"No weights at: {gen_path}  — run training first.")

## 12. Summary

| Component | Detail |
|-----------|--------|
| **Dataset** | 7,200 Brain MRI images (glioma · meningioma · pituitary · notumor) |
| **Preprocessing** | Grayscale resize 64×64, normalize [0,1] |
| **Generator** | z(200) → Dense(16384) → Reshape(4,4,1024) → 4×Conv2DTranspose → 64×64×1 |
| **Critic** | 64×64×1 → 4×Conv2D(128·256·512·1024) → Flatten → Dense(1) |
| **Loss** | Wasserstein + Gradient Penalty (λ=10) |
| **Optimizer** | Adam (lr=5e-5 · β₁=0.5 · β₂=0.9) |
| **Critic/Gen ratio** | 5 critic updates : 1 generator update |
| **Output** | 64×64 grayscale synthetic brain MRI · 10×10 montage |
| **Saved weights** | `saved_weights/gen_{class}.weights.h5` → auto-loaded by `app.py` |

> **App Integration**: Train this notebook → weights are saved → open the web app → select tumor class → click **Generate Synthetic MRI** — the app uses the trained generator automatically.
